# Identify Hurricane Places

Identify BPS places affected by major hurricanes using FEMA Individual Assistance data.

**Criteria:**
- Hurricanes with ≥$150M total IA housing damage
- Counties with >15% IA registration rate
- Places with average monthly permits ≥ 2.0
- Places with continuous permit data for 4 years pre- and 2 years post-disaster

In [2]:
import pandas as pd
import numpy as np
import requests
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

PROJECT_DIR = Path.cwd()
OUTPUT_DIR = PROJECT_DIR / 'output'

MIN_IA_DAMAGE_M = 150     # $M — minimum FEMA IA housing damage for hurricane inclusion
MIN_REG_PCT = 15          # % — minimum county IA registration rate
MIN_AVG_PERMITS = 2.0     # minimum average monthly single-family permits
PRE_YEARS = 4             # years of pre-disaster data required
POST_YEARS = 2            # years of post-disaster data required

STATE_FIPS = {
    '01':'AL','02':'AK','04':'AZ','05':'AR','06':'CA','08':'CO','09':'CT',
    '10':'DE','11':'DC','12':'FL','13':'GA','15':'HI','16':'ID','17':'IL',
    '18':'IN','19':'IA','20':'KS','21':'KY','22':'LA','23':'ME','24':'MD',
    '25':'MA','26':'MI','27':'MN','28':'MS','29':'MO','30':'MT','31':'NE',
    '32':'NV','33':'NH','34':'NJ','35':'NM','36':'NY','37':'NC','38':'ND',
    '39':'OH','40':'OK','41':'OR','42':'PA','44':'RI','45':'SC','46':'SD',
    '47':'TN','48':'TX','49':'UT','50':'VT','51':'VA','53':'WA','54':'WV',
    '55':'WI','56':'WY','72':'PR','78':'VI',
}

print(f"Selection criteria: IA damage >= ${MIN_IA_DAMAGE_M}M, "
      f"county reg rate > {MIN_REG_PCT}%")
print(f"Place criteria: avg permits >= {MIN_AVG_PERMITS}/mo, "
      f"{PRE_YEARS}yr pre + {POST_YEARS}yr post continuous data required")

Selection criteria: IA damage >= $150M, county reg rate > 15%
Place criteria: avg permits >= 2.0/mo, 4yr pre + 2yr post continuous data required


## Load BPS Place Panel

In [5]:
places = pd.read_csv(
    OUTPUT_DIR / 'bps_place_monthly_panel.csv',
    dtype={'place_id': str, 'state_code': str, 'cbsa_code': str, 'county_code': str}
)
places['date'] = pd.to_datetime(
    places['year'].astype(str) + '-' + places['month'].astype(str) + '-01')
places['county_fips'] = (places['state_code'].str.zfill(2)
                         + places['county_code'].str.zfill(3))

print(f"{places.shape[0]:,} rows, {places['place_id'].nunique():,} places, "
      f"{places['date'].min():%Y-%m} to {places['date'].max():%Y-%m}")

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_11856\917635492.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  places = pd.read_csv(


3,278,007 rows, 21,011 places, 2000-01 to 2025-10


## Pull FEMA Hurricane Data

In [8]:
url_decl = "https://www.fema.gov/api/open/v2/DisasterDeclarationsSummaries"

def _pull_pages(url, params, key, delay=0.15):
    """Paginate through FEMA API results."""
    recs, skip = [], 0
    while True:
        params['$skip'] = skip
        try:
            batch = requests.get(url, params=params, timeout=120).json().get(key, [])
        except Exception:
            break
        if not batch:
            break
        recs.extend(batch)
        skip += len(batch)
        time.sleep(delay)
    return recs

print("Pulling hurricane declarations from FEMA API...")
hurr_decl_recs = _pull_pages(url_decl, {
    '$filter': "incidentType eq 'Hurricane'",
    '$select': 'disasterNumber,declarationTitle,incidentBeginDate,state,'
               'fipsStateCode,fipsCountyCode,designatedArea',
    '$top': 1000,
    '$orderby': 'disasterNumber',
}, 'DisasterDeclarationsSummaries')

hurr_decl_df = pd.DataFrame(hurr_decl_recs)
hurr_decl_df['begin_date'] = pd.to_datetime(hurr_decl_df['incidentBeginDate'])
hurr_decl_df['county_fips'] = hurr_decl_df['fipsStateCode'] + hurr_decl_df['fipsCountyCode']

hurr_dns = sorted(hurr_decl_df['disasterNumber'].unique())
dn_titles = hurr_decl_df.groupby('disasterNumber')['declarationTitle'].first().to_dict()
dn_dates = hurr_decl_df.groupby('disasterNumber')['begin_date'].first().to_dict()

print(f"  {len(hurr_decl_df):,} declaration records, {len(hurr_dns)} unique DNs")

Pulling hurricane declarations from FEMA API...
  13,726 declaration records, 454 unique DNs


In [10]:
url_ia = "https://www.fema.gov/api/open/v2/HousingAssistanceOwners"

def pull_disaster_ia(dn):
    recs, skip = [], 0
    while True:
        params = {
            '$filter': f"disasterNumber eq {dn}",
            '$select': 'disasterNumber,state,county,validRegistrations,totalDamage',
            '$top': 1000, '$skip': skip,
        }
        try:
            batch = requests.get(url_ia, params=params, timeout=120).json().get(
                'HousingAssistanceOwners', [])
        except Exception:
            break
        if not batch:
            break
        recs.extend(batch)
        skip += len(batch)
    if not recs:
        return dn, 0, pd.DataFrame()
    df = pd.DataFrame(recs)
    total = df['totalDamage'].sum()
    county = df.groupby(['disasterNumber', 'state', 'county']).agg(
        registrations=('validRegistrations', 'sum'),
        damage=('totalDamage', 'sum')).reset_index()
    return dn, total, county

print(f"Pulling IA data for {len(hurr_dns)} hurricane DNs from FEMA API...")
ia_results = {}
with ThreadPoolExecutor(max_workers=8) as pool:
    futures = {pool.submit(pull_disaster_ia, int(dn)): dn for dn in hurr_dns}
    done = 0
    for f in as_completed(futures):
        dn, total, county_df = f.result()
        ia_results[dn] = (total, county_df)
        done += 1
        if done % 20 == 0:
            print(f"  {done}/{len(hurr_dns)} complete...")
print(f"  {done}/{len(hurr_dns)} complete")

Pulling IA data for 454 hurricane DNs from FEMA API...
  20/454 complete...
  40/454 complete...
  60/454 complete...
  80/454 complete...
  100/454 complete...
  120/454 complete...
  140/454 complete...
  160/454 complete...
  180/454 complete...
  200/454 complete...
  220/454 complete...
  240/454 complete...
  260/454 complete...
  280/454 complete...
  300/454 complete...
  320/454 complete...
  340/454 complete...
  360/454 complete...
  380/454 complete...
  400/454 complete...
  420/454 complete...
  440/454 complete...
  454/454 complete


## Identify Qualifying Hurricanes

In [13]:
def normalize_hurricane_name(title):
    """'HURRICANE KATRINA' -> 'Katrina'"""
    t = title.strip().upper()
    t = t.replace('REMNANTS OF ', '').replace('HURRICANE ', '')
    return t.strip().title()

# Group DNs by hurricane name
hurr_name_map = {}
hurr_groups = {}
for dn in hurr_dns:
    norm = normalize_hurricane_name(dn_titles.get(dn, '?'))
    hurr_name_map[dn] = norm
    hurr_groups.setdefault(norm, []).append(dn)
for k in hurr_groups:
    hurr_groups[k] = sorted(hurr_groups[k])

# Per-hurricane totals (sum across all DNs)
hurr_totals = {}
hurr_dates = {}
for name, dns in hurr_groups.items():
    hurr_totals[name] = sum(ia_results.get(dn, (0, None))[0] for dn in dns)
    valid_dates = [dn_dates[dn] for dn in dns if dn in dn_dates]
    if valid_dates:
        dt = min(valid_dates)
        hurr_dates[name] = dt.tz_localize(None) if hasattr(dt, 'tz') and dt.tz else dt
    else:
        hurr_dates[name] = pd.NaT

# Filter by damage threshold
qualifying = sorted(
    [n for n, t in hurr_totals.items() if t >= MIN_IA_DAMAGE_M * 1e6],
    key=lambda n: hurr_totals[n], reverse=True)

print(f"Grouped {len(hurr_dns)} DNs into {len(hurr_groups)} named hurricanes")
print(f"{len(qualifying)} hurricanes with >= ${MIN_IA_DAMAGE_M}M IA damage:\n")
for name in qualifying:
    print(f"  {name:20s}  {hurr_dates[name]:%Y-%m-%d}  "
          f"${hurr_totals[name]/1e6:>10,.0f}M  DNs: {hurr_groups[name]}")

Grouped 454 DNs into 159 named hurricanes
25 hurricanes with >= $150M IA damage:

  Katrina               2005-08-24  $     6,465M  DNs: [1602, 1603, 1604, 1605, 3212, 3213, 3214, 3231, 3235, 3244]
  Sandy                 2012-10-26  $     2,220M  DNs: [3349, 3350, 3351, 3352, 3353, 3354, 3355, 3356, 3357, 3358, 3359, 3360, 4085, 4086, 4087, 4089, 4090, 4091, 4092, 4093, 4095, 4096, 4097, 4099]
  Harvey                2017-08-23  $     2,182M  DNs: [4332]
  Ian                   2022-09-23  $     1,669M  DNs: [3585, 3586, 4673, 4677]
  Maria                 2017-09-16  $     1,609M  DNs: [3390, 3391, 4339, 4340]
  Ida                   2021-08-26  $     1,329M  DNs: [3569, 3572, 3573, 4611, 4614, 4615, 4618, 4626, 4627, 4629]
  Helene                2024-09-22  $     1,216M  DNs: [3618, 4828, 4829, 4830]
  Ike                   2008-09-05  $       956M  DNs: [1791, 1792, 3293, 3294, 3295]
  Rita                  2005-09-20  $       872M  DNs: [1606, 1607, 3260, 3261]
  Ivan            

## Identify High-Registration Counties

In [16]:
# County populations (Census 2020)
print("Loading county populations (Census 2020)...")
try:
    pop_resp = requests.get(
        "https://api.census.gov/data/2020/dec/pl?get=P1_001N,NAME&for=county:*",
        timeout=60)
    pop_data = pop_resp.json()
    county_pops = {}
    for row in pop_data[1:]:
        cfips = row[2] + row[3]
        county_pops[cfips] = int(row[0])
    print(f"  {len(county_pops)} counties loaded")
except Exception as e:
    print(f"  Census API failed ({e}), estimating from BPS place populations")
    _pi = places.groupby(['state_code', 'county_code']).agg(
        pop=('pop', 'max')).reset_index()
    _pi['cfips'] = _pi['state_code'].str.zfill(2) + _pi['county_code'].str.zfill(3)
    county_pops = _pi.set_index('cfips')['pop'].to_dict()

# Clean county names in declaration data
def _clean_county(s):
    return (s.str.replace(r'\s*\(County\)', ' County', regex=True)
             .str.replace(r'\s*\(Parish\)', ' Parish', regex=True)
             .str.replace(r'\s*\(Borough\)', ' Borough', regex=True)
             .str.replace(r'\s*\(Census Area\)', ' Census Area', regex=True)
             .str.replace(r'\s*\(city\)', ' city', regex=True)
             .str.replace(r'\s*\(City\)', ' City', regex=True)
             .str.replace(r'\s*\(Municipality\)', ' Municipality', regex=True)
             .str.strip())

hurr_decl_df['county_clean'] = _clean_county(hurr_decl_df['designatedArea'])
hurr_fips_lookup = (hurr_decl_df[['state', 'county_clean', 'county_fips']]
                    .drop_duplicates()
                    .set_index(['state', 'county_clean'])['county_fips'].to_dict())

# Build county IA stats for qualifying hurricanes
qualifying_dns = sorted([dn for name in qualifying
                         for dn in hurr_groups[name]
                         if ia_results.get(dn, (0,))[0] > 0])

hurr_county_all = pd.concat(
    [c for dn in qualifying_dns for _, c in [ia_results[dn]] if len(c) > 0],
    ignore_index=True)
hurr_county_all['county_clean'] = _clean_county(hurr_county_all['county'])
hurr_county_all['county_fips'] = hurr_county_all.apply(
    lambda r: hurr_fips_lookup.get((r['state'], r['county_clean'])), axis=1)
hurr_county_all['county_pop'] = hurr_county_all['county_fips'].map(county_pops)
hurr_county_all['reg_pct'] = (hurr_county_all['registrations']
                              / hurr_county_all['county_pop'] * 100)
hurr_county_all['hurricane'] = hurr_county_all['disasterNumber'].map(hurr_name_map)

# Aggregate to hurricane level (same county may appear under multiple DNs)
county_by_hurr = (hurr_county_all
    .groupby(['hurricane', 'county_fips', 'county_clean', 'state'])
    .agg(registrations=('registrations', 'sum'),
         damage=('damage', 'sum'),
         county_pop=('county_pop', 'first'))
    .reset_index())
county_by_hurr['reg_pct'] = (county_by_hurr['registrations']
                              / county_by_hurr['county_pop'] * 100)

# Filter to counties with >15% registration rate
high_reg = county_by_hurr[county_by_hurr['reg_pct'] > MIN_REG_PCT].copy()
high_reg['hurricane_date'] = high_reg['hurricane'].map(hurr_dates)

print(f"Counties with >{MIN_REG_PCT}% registration rate:")
print(f"  {len(high_reg)} county-hurricane pairs across "
      f"{high_reg['hurricane'].nunique()} hurricanes\n")
for name in qualifying:
    subset = high_reg[high_reg['hurricane'] == name]
    if len(subset) > 0:
        print(f"  {name} ({hurr_dates[name]:%Y-%m-%d}):")
        for _, r in subset.sort_values('reg_pct', ascending=False).iterrows():
            st = STATE_FIPS.get(r['county_fips'][:2], r['state'])
            print(f"    {r['county_clean']}, {st}  — {r['reg_pct']:.1f}%  "
                  f"({r['registrations']:,.0f} registrations)")

Loading county populations (Census 2020)...
  Census API failed (HTTPSConnectionPool(host='api.census.gov', port=443): Read timed out. (read timeout=60)), estimating from BPS place populations
Counties with >15% registration rate:
  319 county-hurricane pairs across 23 hurricanes

  Katrina (2005-08-24):
    Lamar County, MS  — 406.8%  (9,755 registrations)
    Perry County, MS  — 307.9%  (3,288 registrations)
    Greene County, MS  — 303.1%  (2,722 registrations)
    Jefferson Davis County, MS  — 302.6%  (3,271 registrations)
    Walthall County, MS  — 273.3%  (4,397 registrations)
    Lawrence County, MS  — 270.2%  (4,245 registrations)
    Jasper County, MS  — 268.7%  (4,799 registrations)
    Franklin County, MS  — 253.0%  (1,136 registrations)
    Washington County, AL  — 213.8%  (2,754 registrations)
    Smith County, MS  — 209.8%  (3,067 registrations)
    Covington County, MS  — 208.0%  (5,378 registrations)
    George County, MS  — 195.5%  (6,100 registrations)
    Clarke Coun

## Find and Filter BPS Places

In [19]:
data_end = places['date'].max()
results = []
skipped_hurricanes = []

for _, county_row in high_reg.iterrows():
    hurr_name = county_row['hurricane']
    hurr_date = county_row['hurricane_date']
    cfips = county_row['county_fips']
    county_name = county_row['county_clean']
    state = county_row['state']
    reg_pct = county_row['reg_pct']

    # Floor hurricane date to month start for BPS alignment
    hurr_month = hurr_date.replace(day=1)
    window_start = hurr_month - pd.DateOffset(months=PRE_YEARS * 12)
    window_end = hurr_month + pd.DateOffset(months=POST_YEARS * 12)
    n_required = PRE_YEARS * 12 + POST_YEARS * 12  # 72 months

    # Skip if window extends beyond available data
    if window_end > data_end + pd.DateOffset(months=1):
        skipped_hurricanes.append(
            f"{hurr_name} ({hurr_date:%Y-%m}) — need data to "
            f"{window_end:%Y-%m}, have to {data_end:%Y-%m}")
        continue

    # Required months in window
    required_months = set(pd.date_range(window_start,
        window_end - pd.DateOffset(months=1), freq='MS'))

    # Find all BPS places in this county
    county_places = places[places['county_fips'] == cfips]
    place_ids = county_places['place_id'].unique()
    st_abbr = STATE_FIPS.get(cfips[:2], state)

    for pid in place_ids:
        pdata = county_places[county_places['place_id'] == pid]
        pname = pdata['place_name'].iloc[0]

        # Check average single-family permits
        avg_permits = pdata['units_1'].mean()
        if avg_permits < MIN_AVG_PERMITS:
            continue

        # Check for continuous data in the required window
        actual_months = set(pdata['date'])
        missing = required_months - actual_months
        if missing:
            continue

        results.append({
            'hurricane_name': hurr_name,
            'hurricane_date': hurr_date.strftime('%Y-%m-%d'),
            'county_name': f"{county_name}, {st_abbr}",
            'county_reg_pct': round(reg_pct, 1),
            'place_id': pid,
            'place_name': pname,
            'avg_permits_month': round(avg_permits, 2),
            'n_months_data': len(pdata),
        })

result_df = pd.DataFrame(results)

if skipped_hurricanes:
    print("Skipped (insufficient post-disaster data):")
    for s in sorted(set(skipped_hurricanes)):
        print(f"  {s}")
    print()

print(f"{len(result_df)} hurricane-place pairs found")
if len(result_df) > 0:
    print(f"  {result_df['hurricane_name'].nunique()} hurricanes, "
          f"  {result_df['place_id'].nunique()} unique places")
result_df.sort_values(['hurricane_date', 'hurricane_name',
                       'county_name', 'place_name']).reset_index(drop=True)

Skipped (insufficient post-disaster data):
  Beryl (2024-07) — need data to 2026-07, have to 2025-10
  Helene (2024-09) — need data to 2026-09, have to 2025-10
  Milton (2024-10) — need data to 2026-10, have to 2025-10

399 hurricane-place pairs found
  19 hurricanes,   254 unique places


,hurricane_name,hurricane_date,county_name,county_reg_pct,place_id,place_name,avg_permits_month,n_months_data
0,Tropical Storm Bonnie And Charley,2004-08-11,"Charlotte County, FL",19.2,12129000,Charlotte County Unincorporated Area,129.91,310
1,Tropical Storm Bonnie And Charley,2004-08-11,"Charlotte County, FL",19.2,12731000,Punta Gorda,8.82,226
2,Tropical Storm Bonnie And Charley,2004-08-11,"Hardee County, FL",23.7,12321000,Hardee County Unincorporated Area,5.51,226
3,Tropical Storm Bonnie And Charley,2004-08-11,"Volusia County, FL",18.1,12189000,Daytona Beach,26.34,310
4,Tropical Storm Bonnie And Charley,2004-08-11,"Volusia County, FL",18.1,12195000,De Land,31.95,310
...,...,...,...,...,...,...,...,...
394,Ian,2022-09-23,"Volusia County, FL",30.0,12189000,Daytona Beach,26.34,310
395,Ian,2022-09-23,"Volusia County, FL",30.0,12195000,De Land,31.95,310
396,Ian,2022-09-23,"Volusia County, FL",30.0,12201100,Deltona,34.36,310
397,Ian,2022-09-23,"Volusia County, FL",30.0,12593000,New Smyrna Beach,16.75,310


## Save Output

In [21]:
out_path = OUTPUT_DIR / 'hurricane_places.csv'
result_df.sort_values(
    ['hurricane_date', 'hurricane_name', 'county_name', 'place_name']
).to_csv(out_path, index=False)
print(f"Saved {len(result_df)} rows to {out_path.name}")

Saved 399 rows to hurricane_places.csv
